# SMQ Research — Colab A100 Runner
**Repo:** `pavannn16/maia-inspired-phi35-smq`

Run cells top-to-bottom on first session. On reconnect, re-run from Cell 1 (takes ~3 min to re-setup).

In [ ]:
# Cell 1 — Verify A100
!nvidia-smi
import torch
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nGPU: {name}  |  Memory: {mem:.1f} GB")
assert 'A100' in name, f'Expected A100, got {name} — reconnect runtime and try again'

In [ ]:
# Cell 2 — Mount Drive (results persist across sessions)
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/smq-results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted. Results will persist to:', RESULTS_DIR)

In [ ]:
# Cell 3 — Clone repo (fresh each session)
import os
REPO = '/content/maia-inspired-phi35-smq'
if not os.path.exists(REPO):
    !git clone https://github.com/pavannn16/maia-inspired-phi35-smq.git {REPO}
else:
    !git -C {REPO} pull
os.chdir(REPO)
!pip install -r requirements.txt -q
print('Done.')

In [ ]:
# Cell 4 — Compile CUDA dequant kernel (required each session)
import os
os.chdir('/content/maia-inspired-phi35-smq/cuda/shared_scale_dequant')
!python setup.py build_ext --inplace
os.chdir('/content/maia-inspired-phi35-smq')
print('CUDA kernel compiled.')

In [ ]:
# Cell 5 — Symlink results → Drive (survives session death)
import os
results_src = '/content/maia-inspired-phi35-smq/results'
results_dst = '/content/drive/MyDrive/smq-results'
if not os.path.islink(results_src):
    !rm -rf {results_src}
    !ln -s {results_dst} {results_src}
print('Symlinked:', results_src, '->', results_dst)

In [ ]:
# Cell 6 — Run unit tests (sanity check before wasting GPU time)
!python -m pytest quant/tests/ -v

---
## Experiment Runs
Run in this order. Each config is independent — if a session dies mid-run, just re-setup (Cells 1–5) and continue from where you left off.

In [ ]:
# Cell 7 — C0: BF16 baseline (run this first)
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m bench.offline_bench \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C0 \
    --repeats 3 \
    --out_dir results

In [ ]:
# Cell 8 — C3: W4 exact scales (SMQ reference)
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m bench.offline_bench \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C3 \
    --repeats 3 \
    --out_dir results

In [ ]:
# Cell 9 — C4: W4 + E5M5 scales (proposed method)
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m bench.offline_bench \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C4 \
    --repeats 3 \
    --out_dir results

In [ ]:
# Cell 10 — C5: scale_mbits sweep {0, 3, 5} (full ablation)
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m bench.offline_bench \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C5 \
    --repeats 3 \
    --out_dir results

In [ ]:
# Cell 11 — C2: bitsandbytes NF4 baseline
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m bench.offline_bench \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C2 \
    --repeats 3 \
    --out_dir results

In [ ]:
# Cell 12 — C6: bitsandbytes double-quant (direct prior-work comparison)
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m bench.offline_bench \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C6 \
    --repeats 3 \
    --out_dir results

In [ ]:
# Cell 13 — C7: W4 + E5M5, all layers (attention + MLP)
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m bench.offline_bench \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C7 \
    --repeats 3 \
    --out_dir results

---
## Quality Evaluation (lm-eval)

In [ ]:
# Cell 14 — lm-eval: C0 BF16 baseline quality
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m eval.lm_eval_runner \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C0 \
    --out_dir results

In [ ]:
# Cell 15 — lm-eval: C4 proposed method quality
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m eval.lm_eval_runner \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C4 \
    --out_dir results

In [ ]:
# Cell 16 — lm-eval: C2 NF4 quality (comparison)
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m eval.lm_eval_runner \
    --config configs/experiment_matrix.yaml \
    --model {MODEL} \
    --config_id C2 \
    --out_dir results

---
## Analysis Scripts (run locally after pulling results — no GPU needed)

In [ ]:
# Cell 17 — Per-layer sensitivity analysis
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m analysis.per_layer_sensitivity \
    --model {MODEL} \
    --scale_mbits_list -1 0 3 5 \
    --out_csv results/aggregate/per_layer_sensitivity.csv \
    --device cuda

In [ ]:
# Cell 18 — Memory audit (empirical vs theoretical savings)
MODEL = 'microsoft/Phi-3.5-mini-instruct'
!python -m analysis.memory_audit \
    --model {MODEL} \
    --group_size 128 \
    --out_csv results/aggregate/memory_audit.csv \
    --device cuda

In [ ]:
# Cell 19 — Aggregate all results into paper_table.csv
!python scripts/aggregate_results.py \
    --summary_dir results/summary \
    --out_csv results/aggregate/paper_table.csv

---
## Push Results to GitHub

In [ ]:
# Cell 20 — Commit and push results
import subprocess, datetime
branch = 'results/' + datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
cmds = [
    ['git', '-C', '/content/maia-inspired-phi35-smq', 'checkout', '-b', branch],
    ['git', '-C', '/content/maia-inspired-phi35-smq', 'add', '-f', 'results/'],
    ['git', '-C', '/content/maia-inspired-phi35-smq', 'commit', '-m', f'Add benchmark results ({branch})'],
    ['git', '-C', '/content/maia-inspired-phi35-smq', 'push', '-u', 'origin', branch],
]
for cmd in cmds:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or r.stderr)